<a href="https://colab.research.google.com/github/imdoamaral/formacao-pnl/blob/main/redes_neurais_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
# from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import confusion_matrix
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Embedding
from google.colab import files

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
files.upload()

In [7]:
spam = pd.read_csv("spam.csv")
spam.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
# fit = cria o modelo
# transform = converte texto para número

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(spam['Category'])
print(y)

[0 0 1 ... 0 0 0]


In [9]:
mensagens = spam['Message'].values
X_train, X_test, y_train, y_test = train_test_split(mensagens, y, test_size=0.3)

In [10]:
# fit apenas no teste, transform em ambos

token = Tokenizer(num_words=1000)
token.fit_on_texts(X_train)

X_train = token.texts_to_sequences(X_train)
X_test = token.texts_to_sequences(X_test)

In [12]:
print(len(X_train))
print(X_train)

3900
[[44, 9, 74, 778, 18, 36, 460, 98, 36, 75, 214, 853, 2, 485, 13, 26, 605, 853, 13, 22, 853], [312, 606, 934, 17, 354, 2, 53, 18], [121, 185, 461, 41, 156, 178, 106, 1, 31, 41, 404, 935], [38, 1, 28, 54, 94, 18, 5, 146], [418, 299, 6, 162, 10], [3, 19, 291, 215, 33, 12, 99, 641, 42, 139, 191, 16, 605, 355, 936, 570, 13, 64, 163], [50, 54], [55, 538, 18, 39, 63, 11, 356, 1, 129, 39, 3, 118, 7, 641, 11, 1, 106, 15, 3, 7, 72, 4, 462, 42, 5, 779, 15, 3], [82, 82, 405, 140, 12, 300], [12, 210, 17, 2, 197, 41, 4, 127, 13, 607, 18, 12, 146, 86, 1, 211, 236, 539, 8, 10], [29, 5, 320, 8, 419, 6, 642, 181, 26, 680, 374, 29, 110, 8, 681, 2, 6, 8, 30, 6, 680, 6, 680, 30, 6, 181, 6, 87, 680], [277, 6, 240, 240, 15, 5, 63, 118, 301], [937, 3, 4, 114], [83, 406, 12, 111, 18, 49], [19, 3, 9, 437], [1, 74, 2, 93], [7, 3, 608, 17, 2, 571, 81, 10, 643, 216, 2, 3, 58, 1, 158, 156, 115, 3, 27, 733, 3, 65, 7, 76, 27, 5, 283], [82, 82, 50, 119, 407, 14, 438], [78, 5, 146, 8, 2, 164, 186, 83], [299, 122, 

In [13]:
X_train = pad_sequences(X_train, padding="post", maxlen=500)
X_test = pad_sequences(X_test, padding="post", maxlen=500)

In [14]:
print(X_train)

[[ 44   9  74 ...   0   0   0]
 [312 606 934 ...   0   0   0]
 [121 185 461 ...   0   0   0]
 ...
 [143 168 212 ...   0   0   0]
 [  6 308  22 ...   0   0   0]
 [232  27   3 ...   0   0   0]]


In [15]:
print(len(token.word_index))

7401


In [31]:
from keras.layers import GlobalMaxPool1D, Input

modelo = Sequential()
# Definindo a entrada com Input e usando GlobalMaxPool1D para melhor extração de características
modelo.add(Input(shape=(500,)))
modelo.add(Embedding(input_dim=len(token.word_index) + 1, output_dim=50))
modelo.add(GlobalMaxPool1D())

modelo.add(Dense(units=10, activation="relu"))
modelo.add(Dropout(0.1))
modelo.add(Dense(units=1, activation="sigmoid"))

In [32]:
# Alterando de mean_squared_error para binary_crossentropy
modelo.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
modelo.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 500, 50)        │       370,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 50)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 10)             │           510 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 370,621 (1.41 MB)

 Trainable params: 370,621 (1.41 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
modelo.fit(X_train, y_train, epochs=20, batch_size=10, validation_data=(X_test, y_test))

Epoch 1/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.8613 - loss: 0.3841 - val_accuracy: 0.8720 - val_loss: 0.2747
Epoch 2/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9497 - loss: 0.1521 - val_accuracy: 0.9785 - val_loss: 0.0739
Epoch 3/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9815 - loss: 0.0633 - val_accuracy: 0.9844 - val_loss: 0.0591
Epoch 4/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9869 - loss: 0.0426 - val_accuracy: 0.9862 - val_loss: 0.0525
Epoch 5/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9921 - loss: 0.0311 - val_accuracy: 0.9868 - val_loss: 0.0402
Epoch 6/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9931 - loss: 0.0243 - val_accuracy: 0.9892 - val_loss: 0.0385
Epoch 7/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9936 - loss: 0.0205 - val_accuracy: 0.9898 - val_loss: 0.0398
Epoch 8/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9956 - loss: 0.0159 - val_accu

In [34]:
loss, accuracy = modelo.evaluate(X_test, y_test)
print("Loss:", loss)
print("Acurácia:", accuracy)

53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9904 - loss: 0.0637
Loss: 0.0636809691786766
Acurácia: 0.9904305934906006


In [35]:
nova_previsao = modelo.predict(X_test)
print(nova_previsao)

53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
[[4.9064915e-11]
 [1.0000000e+00]
 [6.1969549e-05]
 ...
 [1.9659305e-09]
 [6.3808542e-04]
 [8.0382925e-08]]


In [36]:
prev2 = (nova_previsao > 0.5)
print(prev2)

[[False]
 [ True]
 [False]
 ...
 [False]
 [False]
 [False]]


In [37]:
cm = confusion_matrix(y_test, prev2)
print(cm)

[[1455    3]
 [  13  201]]
